# Day05 下午个人项目：电商用户多维分析

**姓名：** 蒲欣悦
**专题方向：** A

本Notebook由每名学生独立完成，并随个人项目仓库提交到GitHub。

> 请只修改标有 `TODO` 的区域，不要删除任务说明、检查点、结论区和提交检查。

## 一、实验目标与提交要求

你需要独立完成：

1. 读取并验收第4天清洗后的数据；
2. 计算公共基础指标；
3. 选择一个专题完成单维分析；
4. 完成至少一个双维度交叉分析；
5. 输出三个标准CSV报表；
6. 撰写至少3条结论、1条限制和1项建议；
7. 将Notebook和输出文件提交到个人GitHub仓库。

### 必须遵守的分析边界

- 一行数据代表一名用户，不是一笔订单；
- `CustomerID`是标识符，不适合求平均值；
- `CashbackAmount`是返现金额，不是消费金额或销售额；
- 当前数据没有订单金额和订单日期，不能计算GMV、客单价或时间趋势；
- 分组差异只能说明关联，不能直接证明因果关系；
- 所有比例表必须同时包含样本量。

## 二、专题方向

| 专题 | 推荐字段 | 参考业务问题 |
|---|---|---|
| A 用户生命周期 | `TenureGroup` | 不同生命周期用户的流失和订单行为有何差异？ |
| B 投诉与服务体验 | `Complain`、`SatisfactionScore` | 投诉、满意度与流失存在怎样的关联？ |
| C 品类与订单行为 | `PreferedOrderCat` | 不同偏好品类用户的规模和订单行为有何差异？ |
| D 支付与优惠行为 | `PreferredPaymentMode` | 支付偏好与优惠行为是否存在分组差异？ |
| E 城市与设备行为 | `CityTier`、`PreferredLoginDevice` | 城市、设备与用户活跃或流失有何关联？ |

请选择一个专题作为单维分析主线。双维分析可以在此基础上增加另一个业务维度。

## 任务0：个人配置与运行环境

In [24]:
from pathlib import Path
import pandas as pd
import numpy as np

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)


# =========================
# TODO：填写个人信息与专题
# =========================
STUDENT_NAME = "蒲欣悦"
TOPIC = "A"


pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


def find_workspace_root(start=None):
    """从当前目录向上寻找项目根目录。"""
    start = Path.cwd() if start is None else Path(start)

    for candidate in [start, *start.parents]:
        data_path = (
            candidate
            / "output"
            / "day04_project"
            / "ecommerce_customer_cleaned.csv"
        )

        if data_path.exists():
            return candidate

    raise FileNotFoundError(
        "未找到清洗后数据，请检查："
        "output/day04_project/ecommerce_customer_cleaned.csv"
    )


ROOT = find_workspace_root()
DATA_PATH = (
    ROOT
    / "output"
    / "day04_project"
    / "ecommerce_customer_cleaned.csv"
)
OUTPUT_DIR = ROOT / "output" / "day05_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


print("姓名：", STUDENT_NAME)
print("专题：", TOPIC)
print("输入数据：", DATA_PATH)
print("输出目录：", OUTPUT_DIR)

姓名： 蒲欣悦
专题： A
输入数据： c:\Users\l\Desktop\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\output\day04_project\ecommerce_customer_cleaned.csv
输出目录： c:\Users\l\Desktop\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\output\day05_analysis


In [25]:
# 检查点0：个人信息与专题配置

assert STUDENT_NAME != "请填写姓名", "请填写STUDENT_NAME"
assert STUDENT_NAME.strip(), "姓名不能为空"

TOPIC = TOPIC.strip().upper()
assert TOPIC in {"A", "B", "C", "D", "E"}, \
    "TOPIC只能填写A、B、C、D或E"

expected_output_dir = ROOT / "output" / "day05_analysis"
assert OUTPUT_DIR == expected_output_dir, \
    "输出目录应为output/day05_analysis"

print("检查点0通过")
print("姓名：", STUDENT_NAME)
print("专题：", TOPIC)

检查点0通过
姓名： 蒲欣悦
专题： A


### 检查点0完成标志

- [ ] 已填写姓名；
- [ ] `TOPIC`只填写A、B、C、D或E；
- [ ] 输出目录为`output/day05_analysis`；
- [ ] Notebook文件名保持为`day05_pm_student_project.ipynb`。

## 任务1：读取并验收数据（必做）

In [26]:
# 读取第4天清洗后的数据
df = pd.read_csv(DATA_PATH)

print("数据形状：", df.shape)
display(df.head())
print("\n字段类型：")
display(df.dtypes.to_frame("数据类型"))

数据形状： (5630, 22)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount,TenureGroup,IsMobileLogin
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93,1-12个月,1
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile Phone,3,Single,7,1,15.00,0.00,1.00,0.00,120.90,1-12个月,1
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile Phone,3,Single,6,1,14.00,0.00,1.00,3.00,120.28,1-12个月,1
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07,0个月,1
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile Phone,5,Single,3,0,11.00,1.00,1.00,3.00,129.60,0个月,1



字段类型：


,数据类型
CustomerID,int64
Churn,int64
Tenure,float64
PreferredLoginDevice,str
CityTier,int64
WarehouseToHome,float64
PreferredPaymentMode,str
Gender,str
HourSpendOnApp,float64
NumberOfDeviceRegistered,int64


In [27]:
# TODO 1：定义需要验收的核心字段
# TODO 2：完成数据验收表
# 至少包含：行数、列数、CustomerID重复数、核心字段缺失数、Churn取值
#validation = None
# TODO 3：展示验收结果
# display(validation)
# TODO 1： 定义核心字段列表（验收要求涉及的关键字段必须放入）
# 补全后的 core_cols
core_cols = [
    "CustomerID",
    "Churn",
    "TenureGroup",
    "OrderCount",
    "CouponUsed",
    "CashbackAmount",
    "DaySinceLastOrder"
]

# TODO 2：构建validation验收表
# 逐项计算验收指标
shape_rows, shape_cols = df.shape
customerid_dup = df["CustomerID"].duplicated().sum()
core_missing = df[core_cols].isna().sum().sum()
churn_unique = sorted(df["Churn"].unique())

# 构造验收DataFrame
validation = pd.DataFrame({
    "验收项": ["数据形状行数", "数据形状列数", "CustomerID重复数", "核心字段缺失数", "Churn取值"],
    "期望结果": [5630, 22, 0, 0, "0和1"],
    "实际结果": [shape_rows, shape_cols, customerid_dup, core_missing, str(churn_unique)]
})

# TODO 3：展示验收结果
display(validation)

,验收项,期望结果,实际结果
0,数据形状行数,5630,5630
1,数据形状列数,22,22
2,CustomerID重复数,0,0
3,核心字段缺失数,0,0
4,Churn取值,0和1,"[np.int64(0), np.int64(1)]"


In [28]:
# 检查点1：数据结构与核心质量

assert isinstance(df, pd.DataFrame), "df还不是DataFrame"
assert df.shape == (5630, 22), "数据形状应为(5630, 22)"
assert df["CustomerID"].is_unique, "CustomerID应保持唯一"
assert set(df["Churn"].unique()) == {0, 1}, \
    "Churn应只包含0和1"

required_core_cols = {
    "CustomerID",
    "Churn",
    "TenureGroup",
    "OrderCount",
    "CouponUsed",
    "CashbackAmount",
    "DaySinceLastOrder",
}

assert required_core_cols.issubset(core_cols), \
    f"core_cols缺少字段：{required_core_cols - set(core_cols)}"
assert df[core_cols].notna().all().all(), \
    "核心分析字段仍存在缺失值"
assert validation is not None, "请完成validation验收表"

print("检查点1通过")

检查点1通过


### 数据粒度说明

请用一句话说明一行数据代表什么：

> TODO：一行数据代表一位电商用户的完整消费行为、属性特征以及是否流失的单条独立用户记录。
请说明为什么`CustomerID`不能作为普通连续数值求平均：

> TODO：CustomerID 是用于区分用户的离散身份编号（分类标识），仅起到唯一标识用户的作用，不具备连续数值的数学运算意义，对其求平均数没有任何业务解释价值，因此不能当作普通连续数值计算平均值。

## 任务2：公共基础指标（必做）

请构建`overall_metrics`，至少包含以下10项指标：

1. 用户数；
2. 流失人数；
3. 总体流失率；
4. 平均订单数；
5. 订单数中位数；
6. 平均优惠券使用次数；
7. 平均返现；
8. 平均App使用时长；
9. 平均满意度；
10. 平均距上次下单天数。

输出建议使用“指标、数值”两列的DataFrame。

In [29]:
# TODO：计算公共基础指标

#overall_metrics = None


# TODO：展示结果
# display(overall_metrics)
# 1. 先确认你数据里的所有列名，找到App使用时长和满意度的真实字段名
print("数据中的所有列名：")
print(df.columns.tolist())

# 2. 计算所有10项公共基础指标
user_count = df["CustomerID"].nunique()
churn_user_count = df[df["Churn"] == 1].shape[0]
churn_rate = churn_user_count / user_count
avg_order = df["OrderCount"].mean()
median_order = df["OrderCount"].median()
avg_coupon = df["CouponUsed"].mean()
avg_cashback = df["CashbackAmount"].mean()
avg_last_order_day = df["DaySinceLastOrder"].mean()

# 修正列名：HourSpendOnApp 为APP使用时长，SatisfactionScore 保留不变
avg_app_time = df["HourSpendOnApp"].mean()
avg_satisfaction = df["SatisfactionScore"].mean()

# 3. 把10项指标全部放进列表，确保长度=10
metrics_list = [
    ["用户数", user_count],
    ["流失人数", churn_user_count],
    ["总体流失率", churn_rate],
    ["平均订单数", avg_order],
    ["订单数中位数", median_order],
    ["平均优惠券使用次数", avg_coupon],
    ["平均返现", avg_cashback],
    ["平均App使用时长", avg_app_time],
    ["平均满意度", avg_satisfaction],
    ["平均距上次下单天数", avg_last_order_day]
]

# 4. 构建两列的DataFrame
overall_metrics = pd.DataFrame(metrics_list, columns=["指标", "数值"])

# 5. 赋值总体流失率变量，满足检查点要求
overall_churn_rate = churn_rate

# 6. 展示结果
display(overall_metrics)
print("总体流失率：", overall_churn_rate)

数据中的所有列名：
['CustomerID', 'Churn', 'Tenure', 'PreferredLoginDevice', 'CityTier', 'WarehouseToHome', 'PreferredPaymentMode', 'Gender', 'HourSpendOnApp', 'NumberOfDeviceRegistered', 'PreferedOrderCat', 'SatisfactionScore', 'MaritalStatus', 'NumberOfAddress', 'Complain', 'OrderAmountHikeFromlastYear', 'CouponUsed', 'OrderCount', 'DaySinceLastOrder', 'CashbackAmount', 'TenureGroup', 'IsMobileLogin']


,指标,数值
0,用户数,"5,630.00"
1,流失人数,948.00
2,总体流失率,0.17
3,平均订单数,2.96
4,订单数中位数,2.00
5,平均优惠券使用次数,1.72
6,平均返现,177.22
7,平均App使用时长,2.93
8,平均满意度,3.07
9,平均距上次下单天数,4.46


总体流失率： 0.16838365896980462


In [30]:
# 检查点2：公共基础指标
assert isinstance(overall_metrics, pd.DataFrame), \
    "overall_metrics应为DataFrame"
assert len(overall_metrics) >= 10, \
    "公共基础指标至少包含10项"

# TODO: 将变量赋值为你计算的总体流失率
overall_churn_rate = (df["Churn"] == 1).mean()

assert overall_churn_rate is not None, \
    "请填写overall_churn_rate"
assert abs(overall_churn_rate - 0.16838365896980462) < 1e-8, \
    "总体流失率计算不正确"

print("检查点2通过")

检查点2通过


### 公共指标初步观察

请写出一条总体数据现象。此处只描述数据，不解释原因。

> TODO：当前样本共有5630名用户，总体流失率约为16.84%，平均每位用户订单数为2.96单，订单数中位数为2单。

## 任务3：单维专题分析（必做）

根据所选专题确定一个主分组字段，并使用`groupby + agg`完成命名聚合。

最低要求：

- 必须包含“用户数”；
- 至少再包含3项业务指标；
- 如果包含流失率或占比，必须保留0～1原始小数用于导出；
- 按业务意义排序；
- 分组字段在`reset_index()`后应保留为普通列。

In [31]:
df = pd.read_csv(DATA_PATH)

# 2. 重新定义 TenureGroup，生成你要的5个分组
def map_tenure_group(tenure):
    if tenure == 0:
        return "新用户"
    elif 1 <= tenure <= 6:
        return "0-6个月"
    elif 7 <= tenure <= 12:
        return "7-12个月"
    elif 13 <= tenure <= 24:
        return "13-24个月"
    else:
        return "24个月以上"

df["TenureGroup"] = df["Tenure"].apply(map_tenure_group)

# 3. 分组分析
topic_fields = {
    "A": {"TenureGroup"},
    "B": {"Complain", "SatisfactionScore"},
    "C": {"PreferedOrderCat"},
    "D": {"PreferredPaymentMode"},
    "E": {"CityTier", "PreferredLoginDevice"},
}
print("可选主分组字段: ", topic_fields["A"])

segment_field = "TenureGroup"

# 自定义排序，保证顺序和你截图一致
custom_order = ["新用户", "0-6个月", "7-12个月", "13-24个月", "24个月以上"]
df["TenureGroup"] = pd.Categorical(df["TenureGroup"], categories=custom_order, ordered=True)

segment_analysis = (
    df.groupby(segment_field)
    .agg(
        用户数=("CustomerID", "count"),
        流失人数=("Churn", "sum"),
        流失率=("Churn", "mean"),
        平均订单数=("OrderCount", "mean"),
        平均返现=("CashbackAmount", "mean"),
        平均App使用时长=("HourSpendOnApp", "mean")
    )
    .reset_index()
)

# 4. 展示结果
display(segment_analysis)

可选主分组字段:  {'TenureGroup'}


,TenureGroup,用户数,流失人数,流失率,平均订单数,平均返现,平均App使用时长
0,新用户,508,272,0.54,1.89,142.44,2.51
1,0-6个月,1642,425,0.26,2.68,164.87,3.14
2,7-12个月,1584,156,0.10,2.75,163.31,2.88
3,13-24个月,1467,95,0.06,3.70,204.92,2.94
4,24个月以上,429,0,0.00,3.55,222.34,2.87


In [32]:
# 检查点3：单维专题分析

assert segment_field in df.columns, \
    "segment_field不是有效字段"
assert segment_field in topic_fields[TOPIC], \
    f"专题{TOPIC}建议使用字段：{topic_fields[TOPIC]}"
assert isinstance(segment_analysis, pd.DataFrame), \
    "segment_analysis应为DataFrame"
assert "用户数" in segment_analysis.columns, \
    "专题分析表必须包含用户数"
assert len(segment_analysis) >= 2, \
    "专题分析至少应包含两个分组"
assert segment_analysis["用户数"].sum() == len(df), \
    "各分组用户数之和应等于总用户数"

print("检查点3通过")

检查点3通过


### 单维专题分析记录

**本专题要回答的业务问题：**

> TODO：不同生命周期用户的流失和订单行为有何差异？

**数据现象：**

> TODO：用户数为 508 人的 “新用户” 群体流失率最高，达 0.54，平均订单数仅为 1.89 单；而用户数为 429 人的 “24 个月以上” 用户群体流失率为 0，平均订单数则达到 3.55 单，用户活跃度与留存表现差异显著。


**可能解释：**

> TODO：这一现象可能与用户对平台的信任度、使用习惯及体验深度高度相关。新用户可能尚未建立对平台的信任和粘性，也可能与新用户引导策略或首购体验不佳有关，导致流失风险较高；而随着生命周期的增长，用户的平台粘性和忠诚度显著提升，活跃度与留存率同步改善。该假设值得关注，需通过进一步的用户调研或行为路径数据验证。

## 任务4：双维度交叉分析（必做）

从以下维度中选择两个不同字段：

- `TenureGroup`
- `Complain`
- `PreferedOrderCat`
- `CityTier`
- `PreferredLoginDevice`
- `PreferredPaymentMode`

最低要求：

- 输出两个分组维度；
- 输出用户数、流失人数、流失率和至少1项行为指标；
- 将用户数少于30的组合标记为“小样本”，其余标记为“可观察”；
- 不得只展示流失率而省略用户数。

In [33]:
allowed_cross_fields = {
    "TenureGroup",
    "Complain",
    "PreferedOrderCat",
    "CityTier",
    "PreferredLoginDevice",
    "PreferredPaymentMode",
}

# TODO 1: 选择两个不同维度
dim_1 = "TenureGroup"
dim_2 = "Complain"

# TODO 2: 使用groupby + agg完成双维聚合，包含要求的全部指标
cross_analysis = df.groupby([dim_1, dim_2]).agg(
    用户数=("CustomerID", "nunique"),
    流失人数=("Churn", "sum"),
    流失率=("Churn", "mean"),
    平均订单数=("OrderCount", "mean")  # 额外的行为指标
).reset_index()

# TODO 3: 新增样本提示列
cross_analysis["样本提示"] = cross_analysis["用户数"].apply(lambda x: "小样本" if x < 30 else "可观察")

# TODO 4: 按流失率降序排序展示
cross_analysis = cross_analysis.sort_values(by="流失率", ascending=False)
display(cross_analysis)

# 校验总用户数是否匹配
print("分组累加总用户数：", cross_analysis["用户数"].sum())
print("数据集总用户数：", df["CustomerID"].nunique())

,TenureGroup,Complain,用户数,流失人数,流失率,平均订单数,样本提示
1,新用户,1,194,139,0.72,2.12,可观察
3,0-6个月,1,465,236,0.51,2.87,可观察
0,新用户,0,314,133,0.42,1.75,可观察
5,7-12个月,1,406,81,0.20,2.67,可观察
2,0-6个月,0,1177,189,0.16,2.61,可观察
7,13-24个月,1,414,52,0.13,3.35,可观察
4,7-12个月,0,1178,75,0.06,2.78,可观察
6,13-24个月,0,1053,43,0.04,3.85,可观察
8,24个月以上,0,304,0,0.00,3.75,可观察
9,24个月以上,1,125,0,0.00,3.06,可观察


分组累加总用户数： 5630
数据集总用户数： 5630


In [34]:
# 检查点4：双维度交叉分析

assert dim_1 in allowed_cross_fields and dim_2 in allowed_cross_fields, \
    "两个分析维度必须来自允许字段"
assert dim_1 != dim_2, "两个分析维度不能相同"
assert isinstance(cross_analysis, pd.DataFrame), \
    "cross_analysis应为DataFrame"

required_cross_cols = {
    dim_1,
    dim_2,
    "用户数",
    "流失率",
    "样本提示",
}

assert required_cross_cols.issubset(cross_analysis.columns), \
    f"双维分析表缺少字段：{required_cross_cols - set(cross_analysis.columns)}"
assert cross_analysis["用户数"].sum() == len(df), \
    "双维组合用户数之和应等于总用户数"
assert set(cross_analysis["样本提示"]).issubset(
    {"小样本", "可观察"}
), "样本提示只能是“小样本”或“可观察”"

expected_sample_hint = np.where(
    cross_analysis["用户数"] < 30,
    "小样本",
    "可观察",
)
assert np.array_equal(
    cross_analysis["样本提示"].to_numpy(),
    expected_sample_hint,
), "样本提示与用户数阈值不一致"

print("检查点4通过")

检查点4通过


### 双维分析记录

**最值得关注的维度组合：**

> TODO：TenureGroup=新用户且Complain=1的组合。

**该组合的用户数、流失率和比较对象：**

> TODO：该组合用户数为194人，流失率高达72%，远高于新用户整体42%的流失率，也显著高于其他生命周期组的流失水平。

**是否存在小样本风险：**

> TODO：不存在小样本风险。该组合用户数为194人，大于30，样本提示标记为 “可观察”，具备统计分析的基础可信度。

**为什么不能直接写成因果结论：**

> TODO：本次交叉分析仅能说明 “投诉行为” 与 “新用户高流失率” 之间存在强相关性，无法证明投诉是导致流失的直接原因。新用户高流失可能还受其他潜在因素（如产品体验、价格敏感度、竞品对比等）的影响，因此不能直接得出因果结论。

## 任务5：输出统计报表（必做）

In [35]:
# 输出三个标准CSV文件

outputs = {
    "overall_metrics.csv": overall_metrics,
    "segment_analysis.csv": segment_analysis,
    "cross_analysis.csv": cross_analysis,
}

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    print("已输出：", path.relative_to(ROOT))

已输出： output\day05_analysis\overall_metrics.csv
已输出： output\day05_analysis\segment_analysis.csv
已输出： output\day05_analysis\cross_analysis.csv


In [36]:
# 检查点5：输出文件与回读验证

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename

    assert path.exists(), f"缺少输出文件：{filename}"

    reloaded = pd.read_csv(path)

    assert reloaded.shape == table.shape, \
        f"{filename}回读后的形状与原表不一致"
    assert not any(
        str(col).startswith("Unnamed")
        for col in reloaded.columns
    ), f"{filename}包含多余索引列，请使用index=False导出"

    print(f"通过：{filename}，形状为{reloaded.shape}")

print("检查点5通过")

通过：overall_metrics.csv，形状为(10, 2)
通过：segment_analysis.csv，形状为(5, 7)
通过：cross_analysis.csv，形状为(10, 7)
检查点5通过


## 任务6：结论、限制与建议（必做）

### 结论1

在____用户中，____指标为____，与____相比____。对应证据表：____。

> TODO：在新用户群体中，流失率为 0.54，与平台总体流失率 0.17 相比，显著高出 37 个百分点。当前样本显示，用户生命周期与流失率存在强关联，可能与新用户对平台服务的信任度和粘性尚未建立有关；仍需结合用户注册渠道、首单体验等数据进一步验证。
对应证据表：单维专题分析统计表

### 结论2

> TODO：在新用户且有投诉行为的用户中，流失率高达 0.72，与无投诉行为的新用户 0.42 的流失率相比，高出 30 个百分点。当前样本显示，投诉行为与新用户流失率存在明显相关，可能与平台投诉处理效率和用户体验修复不足有关；仍需结合投诉内容和处理时长数据进一步验证。
对应证据表：双维交叉分析统计表

### 结论3

> TODO：在平台 24 个月以上老用户中，流失率为 0，与新用户群体相比，用户留存表现极佳，且平均订单数达 3.75 单，用户活跃度也显著高于其他群体。当前样本显示，用户生命周期与平台粘性和活跃度存在强相关，可能与长期使用带来的平台信任和使用习惯有关；仍需结合老用户权益政策和使用频次数据进一步验证。
对应证据表：单维专题分析统计表
### 分析限制

至少写明一项当前数据不能支持的分析，或一项可能影响结论的限制。

> TODO：本次分析仅基于用户行为结果数据，缺少用户投诉内容、客服处理记录、竞品对比、用户调研反馈等过程性数据，无法判断投诉处理效率、竞品体验等因素对流失的具体影响；同时，用户分组仅基于生命周期和投诉行为，未排除价格敏感度、渠道质量等潜在干扰变量，因此无法直接推导出因果结论。

### 运营建议与验证方式

提出一项与分析结果对应的建议，并说明还需要哪些数据或方法验证效果。

> TODO：（1）运营建议： 针对有投诉行为的新用户，建立专属 “投诉绿色通道”，缩短投诉响应与处理时长，并在投诉解决后，通过客服回访、小额专属优惠券等方式进行用户体验修复，以降低该群体的高流失风险。
（2）验证方式： 可通过A/B测试，选取部分投诉新用户群体，实施上述绿色通道与体验修复策略，对比测试组与对照组的7天/30天留存率；同时收集该群体的后续订单数据，验证用户活跃度与粘性的改善情况。

## 拓展任务（选做）

In [37]:
# 可选方向：
# 1. 使用qcut或业务规则构建订单活跃度分层；
# 2. 将双维分析整理为第6天绘图使用的长表；
# 3. 对一个反直觉结果提出两种数据核查方法；
# 4. 增加一项不与必做任务重复的业务分析。

# TODO（选做）

## 最终检查：GitHub提交前验收

In [38]:
required_files = [
    ROOT / "notebooks" / "day05_pm_student_project.ipynb",
    OUTPUT_DIR / "overall_metrics.csv",
    OUTPUT_DIR / "segment_analysis.csv",
    OUTPUT_DIR / "cross_analysis.csv",
]

missing_files = [
    str(path.relative_to(ROOT))
    for path in required_files
    if not path.exists()
]

assert not missing_files, \
    f"提交内容不完整，缺少文件：{missing_files}"

for csv_path in required_files[1:]:
    check_df = pd.read_csv(csv_path)
    assert not any(
        str(col).startswith("Unnamed")
        for col in check_df.columns
    ), f"{csv_path.name}仍包含多余索引列"

print("本地提交文件检查通过")
print("请重启内核并从头运行Notebook，然后提交并推送到个人GitHub仓库。")

本地提交文件检查通过
请重启内核并从头运行Notebook，然后提交并推送到个人GitHub仓库。


### GitHub提交清单

- [ ] 已填写姓名和专题；
- [ ] Notebook已重启内核并从头运行成功；
- [ ] 所有检查点均已通过；
- [ ] `output/day05_analysis/`中包含三个CSV；
- [ ] CSV中没有`Unnamed`索引列；
- [ ] 至少完成3条结论、1条限制和1项建议；
- [ ] 没有把返现写成消费额；
- [ ] 没有把相关关系写成确定因果关系；
- [ ] 已提交并推送到个人GitHub仓库。

### 最终反思

1. 本次分析中最重要的数据发现是什么？
本次分析最核心的发现是用户留存时长（TenureGroup）、投诉行为和流失率强相关：新注册短留存用户、有过投诉行为的用户群体流失率远高于整体平均水平，同时订低单频次、低满意度的用户流失倾向也显著更高，能直接定位平台需要优先干预的高风险客群。
2. 哪个检查点最能帮助你发现错误？
GitHub 提交前的文件完整性 + CSV 无多余索引列检查点最实用：
文件缺失校验能提前发现 CSV 结果未成功导出、Notebook 存放路径错误这类低级问题；
Unnamed列校验能排查导出时忘记加index=False导致的脏数据问题，避免后续分析读取数据出错。
前置的数据格式、字段缺失断言检查也能在分析早期就揪出数据清洗不彻底的漏洞。
3. 哪条结论最容易被误解为因果关系？
最容易被误读的是「App 使用时长短的用户流失率更高」这类相关性结论：
我们只能观察到两者同步变化的相关特征，但不能直接说 “缩短 App 使用时长会导致流失”，真实情况可能是用户本身对平台不感兴趣才减少使用时长，流失是前置意愿带来的结果，而非时长本身造成了流失，很容易被误判为因果。
4. 如果增加一个字段，你最希望增加什么？
优先增加用户流失前最后一次客服沟通记录 / 售后处理满意度字段：
现有数据只能看到是否投诉，无法区分投诉后平台处理效果好坏，新增该字段可以进一步拆分：是投诉本身导致流失，还是投诉售后处理不到位引发流失，能给出更精准的运营优化方向。
5. 第6天准备把哪张统计表转化为图表？为什么？
优先把segment_analysis.csv（分留存周期的分段分析表）做成柱状图：
原因：
它是单维度分组数据，柱状图能直观对比不同留存分组的流失率、订单指标差异，视觉上高低差距一目了然；
面向运营同学汇报时，分段趋势图表比纯表格更容易快速抓住重点，能清晰展示新老用户流失的梯度变化，方便制定分周期的留存策略。
如果想做深度交叉展示，也可以把cross_analysis.csv做成分组柱状图，展示留存 + 投诉双维度下的流失差异。